In [5]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.band_causal_tcn import BandCausalTCNModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import test_toolkit as ts
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "band_causal_tcn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


In [2]:
import time

scorer = VolatilityScorer()

best_model = BandCausalTCNModel(
    look_back=14,
    num_levels=5,
    channels=64,
    kernel_size=3,
    num_convs=2,
    dropout=0.232,
    learning_rate=0.0072,
    weight_decay=6.0e-04,
    J=3,
    wavelet="db4",
    band_kernel_size=1,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

fit_time_sec = end_time - start_time
print(f"Entrenamiento completado en {fit_time_sec:.2f} s")
print(
    f"Converged: {best_model.diagnostics['converged']} | "
    f"Epochs: {best_model.diagnostics['epochs_run']} | "
    f"Final val_loss: {best_model.history['val_loss'][-1]:.6f}"
)

Entrenamiento completado en 114.03 s
Converged: True | Epochs: 26 | Final val_loss: 0.439756


In [3]:
import numpy as np
from src.utils import trainer_toolkit as tt

history = best_model.get_history()
diagnostics = best_model.get_diagnostics()

train_loss = np.asarray(history.get("train_loss", []), dtype=float)
val_loss = np.asarray(history.get("val_loss", []), dtype=float)

assert len(train_loss) > 0 and len(val_loss) > 0, "El historial de entrenamiento está vacío."
assert np.isfinite(train_loss).all(), "train_loss contiene NaN o Inf."
assert np.isfinite(val_loss).all(), "val_loss contiene NaN o Inf."

print(
    f"Épocas ejecutadas: {diagnostics.get('epochs_run', len(train_loss))} "
    f"| Convergió: {diagnostics.get('converged')}"
)
print(
    f"Train QLIKE inicial -> final/mínimo: {train_loss[0]:.6f} -> {train_loss[-1]:.6f} "
    f"(min={train_loss.min():.6f})"
)
print(
    f"Val   QLIKE inicial -> final/mínimo: {val_loss[0]:.6f} -> {val_loss[-1]:.6f} "
    f"(min={val_loss.min():.6f})"
)

max_epochs_plotted = len(train_loss)

tt.plot_learning_curve(
    history=history,
    title="Curva de Aprendizaje (Band Causal TCN)",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 5),
    xlim=(0, max(10, max_epochs_plotted)),
    y_step=0.5,
    x_step=5,
    legend_bbox=(0.95, 0.95),
)

Épocas ejecutadas: 26 | Convergió: True
Train QLIKE inicial -> final/mínimo: 454.543746 -> 0.385569 (min=0.385569)
Val   QLIKE inicial -> final/mínimo: 1.849562 -> 0.439756 (min=0.362595)


In [4]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.dataset import TimeSeriesDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_eff = rs_train * best_model.target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_eff,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=best_model.batch_size,
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(rs_train[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Band Causal TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Band Causal TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

In [7]:
exec_dir = Path(r"C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\results\band_causal_tcn\exec_1")
csv_path = exec_dir / "BandCausalTCNModel_results.csv"

if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("BandCausalTCNModel*_results.csv")) or sorted(
        exec_dir.glob("*_results.csv")
    )
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_band_tcn = pd.read_csv(csv_path)

if "learning_rate" in df_band_tcn.columns:
    df_band_tcn["lr"] = df_band_tcn["learning_rate"].apply(lambda x: f"{x:.4f}")
if "weight_decay" in df_band_tcn.columns:
    df_band_tcn["wd"] = df_band_tcn["weight_decay"].apply(lambda x: f"{x:.1e}")
if "dropout" in df_band_tcn.columns:
    df_band_tcn["drop_fmt"] = df_band_tcn["dropout"].apply(lambda x: f"{x:.3f}")

candidatas_agrupar = [
    "look_back",
    "num_levels",
    "channels",
    "kernel_size",
    "num_convs",
    "J",
    "wavelet",
    "band_kernel_size",
    "drop_fmt",
    "lr",
    "wd",
]
cols_agrupar = [c for c in candidatas_agrupar if c in df_band_tcn.columns]

metric_col = "qlike" if "qlike" in df_band_tcn.columns else "val_loss"

df_agg = (
    df_band_tcn.groupby(cols_agrupar, as_index=False)[metric_col]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

renombres = {
    "look_back": "Lookback",
    "num_levels": "Levels",
    "channels": "Channels",
    "kernel_size": "Kernel",
    "num_convs": "Convs",
    "J": "J",
    "wavelet": "Wavelet",
    "band_kernel_size": "Band Kernel",
    "drop_fmt": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay",
}

df_top5 = (
    df_agg.sort_values(by="mean", ascending=True)
    .head(5)
    .rename(columns={k: v for k, v in renombres.items() if k in df_agg.columns})
)

cols_finales = [
    v for v in renombres.values() if v in df_top5.columns
] + ["QLIKE"]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Levels,Channels,Kernel,Convs,J,Wavelet,Band Kernel,Dropout,LR,Weight Decay,QLIKE
7,4,128,2,3,2,db4,1,0.248,0.0010,9.2e-04,0.463 ± 0.154
7,3,64,2,3,1,db4,1,0.177,0.0004,6.4e-04,0.535 ± 0.276
7,2,128,3,2,2,db4,2,0.222,0.0004,3.8e-04,0.732 ± 0.735
14,5,128,2,2,2,db4,1,0.082,0.0012,3.3e-06,3.034 ± 5.631
7,3,32,3,1,1,sym4,1,0.073,0.0046,1.3e-05,4.498 ± 9.090
